In [5]:
from scipy import signal as sp
import numpy as np
import matplotlib.pyplot as plt

# Ej 2: Síntesis de instrumentos

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Ej 2.1: PSOLA

In [7]:
import numpy as np
import scipy.io.wavfile as wav
import scipy.signal as signal
import matplotlib.pyplot as plt

def estimar_f0_autocorrelacion(senal, fs):
    """Estima la frecuencia fundamental (f0) y el período en muestras."""
    # Autocorrelación completa y nos quedamos con la mitad derecha
    corr = signal.correlate(senal, senal, mode='full')
    corr = corr[len(corr)//2:]

    # Rango de búsqueda típico para instrumentos (ej. 60Hz a 1000Hz)
    min_lag = int(fs / 1000.0)
    max_lag = int(fs / 60.0)

    # Asegurarnos de no exceder la longitud del arreglo
    max_lag = min(max_lag, len(corr))

    # Encontrar el pico de autocorrelación en ese rango
    pico_lag = np.argmax(corr[min_lag:max_lag]) + min_lag
    f0 = fs / pico_lag

    return f0, int(pico_lag)

def detectar_pitch_marks(senal, periodo_muestras):
    """Encuentra los picos espaciados a 1 período de distancia."""
    # distance=periodo*0.8 evita detectar picos secundarios falsos (armónicos)
    picos, _ = signal.find_peaks(senal, distance=periodo_muestras * 0.8)
    return picos

### B

In [8]:
def psola_time_stretch(senal, pitch_marks, factor_estiramiento, fs):
    """
    factor_estiramiento > 1: Alarga el audio.
    factor_estiramiento < 1: Acorta el audio.
    """
    nueva_longitud = int(len(senal) * factor_estiramiento)
    senal_salida = np.zeros(nueva_longitud)

    # Nuevas marcas de tiempo en la señal de salida
    nuevos_pitch_marks = np.arange(0, nueva_longitud, pitch_marks[1] - pitch_marks[0])

    for nuevo_pm in nuevos_pitch_marks:
        # Encontrar el pitch mark original más cercano usando el factor
        pm_original_ideal = nuevo_pm / factor_estiramiento
        idx_cercano = np.argmin(np.abs(pitch_marks - pm_original_ideal))
        pm_real = pitch_marks[idx_cercano]

        # Tamaño de ventana: usualmente 2 o 3 veces el período fundamental
        periodo = pitch_marks[1] - pitch_marks[0] if len(pitch_marks) > 1 else int(fs/440)
        mitad_ventana = periodo

        inicio_orig = max(0, pm_real - mitad_ventana)
        fin_orig = min(len(senal), pm_real + mitad_ventana)

        inicio_nuevo = max(0, int(nuevo_pm) - mitad_ventana)
        fin_nuevo = inicio_nuevo + (fin_orig - inicio_orig)

        # Asegurar que no nos pasamos de los límites del arreglo de salida
        if fin_nuevo > nueva_longitud:
            fin_orig -= (fin_nuevo - nueva_longitud)
            fin_nuevo = nueva_longitud

        # Ventana de Hann para suavizar (inciso d pide experimentar con ventanas)
        ventana = np.hanning(fin_orig - inicio_orig) ######################################################

        # Segmentación y Overlap-Add
        segmento = senal[inicio_orig:fin_orig] * ventana
        senal_salida[inicio_nuevo:fin_nuevo] += segmento

    return senal_salida

def pitch_shift(senal, pitch_marks, semitonos, fs):
    """Cambia el pitch combinando PSOLA y remuestreo."""
    # 1 semitono = 2**(1/12)
    factor_frecuencia = 2.0 ** (semitonos / 12.0)

    # 1. Estirar en el tiempo con PSOLA (inversamente proporcional a la frecuencia)
    factor_stretch = factor_frecuencia
    senal_estirada = psola_time_stretch(senal, pitch_marks, factor_stretch, fs)

    # 2. Remuestrear para restaurar la duración y cambiar el pitch real
    nueva_cantidad_muestras = int(len(senal_estirada) / factor_frecuencia)
    senal_shifteada = signal.resample(senal_estirada, nueva_cantidad_muestras)

    return senal_shifteada

### C

In [9]:
!pip install mido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.0 MB/s eta 0:00:00


In [10]:
import mido

def extraer_notas_midi(ruta_midi):
    midi_file = mido.MidiFile(ruta_midi)
    notas = []
    notas_activas = {}

    # IMPORTANTE: tiempo_actual ahora sí acumulará segundos reales
    tiempo_actual = 0

    # CORRECCIÓN CLAVE: Iterar sobre 'midi_file' directamente (en lugar de midi_file.tracks[0])
    # Esto hace que mido fusione las pistas y convierta 'msg.time' a segundos flotantes automáticamente.
    for msg in midi_file:
        tiempo_actual += msg.time # Ahora msg.time viene en segundos (ej: 0.5 segundos)

        if msg.type == 'note_on' and msg.velocity > 0:
            notas_activas[msg.note] = tiempo_actual

        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            if msg.note in notas_activas:
                inicio = notas_activas[msg.note]
                duracion = tiempo_actual - inicio
                notas.append({
                    'pitch': msg.note,
                    'inicio': inicio,
                    'duracion': duracion
                })
                del notas_activas[msg.note]

    return notas, tiempo_actual

In [11]:
import numpy as np
import soundfile as sf

# Supongamos que re-utilizamos la función pitch_shift(audio_mono, pitch_marks, semitonos, fs) del paso anterior.
# Nota: Asegúrate de tener cargadas tus funciones del Inciso A y B (detección de pitch marks y PSOLA).

def sintetizar_midi_a_audio(ruta_midi, ruta_sample_base, nota_base_midi=60):
    """
    - ruta_sample_base: Ruta al archivo .aif de base (ej: el C4 de guitarra o flauta).
    - nota_base_midi: El número MIDI de ese sample base (C4 = 60).
    """
    # 1. Extraer los eventos del MIDI
    lista_notas, duracion_total_cancion = extraer_notas_midi(ruta_midi)

    # 2. Cargar el sample base .aif (en Mono)
    audio_stereo, fs = sf.read(ruta_sample_base)
    if len(audio_stereo.shape) > 1:
        audio_base = np.mean(audio_stereo, axis=1)
    else:
        audio_base = audio_stereo

    # recorto solo una nota
    audio_base = audio_base[int(fs*0.075):int(fs*3)]

    # 3. Analizar el sample base una sola vez (Inciso A)
    # Calculamos su período fundamental y sus pitch marks base
    f0_base, periodo_base = estimar_f0_autocorrelacion(audio_base, fs)
    pitch_marks_base = detectar_pitch_marks(audio_base, periodo_base)

    # 4. Crear el lienzo de salida (un array de ceros gigante para toda la canción)
    muestras_totales = int((duracion_total_cancion + 2.0) * fs) # +2s de margen por el decaimiento
    audio_final = np.zeros(muestras_totales)

    # 5. Iterar nota por nota y mezclarlas (Overlap-Add a nivel de canción)
    for nota in lista_notas:
        # Calcular cuántos semitonos debemos mover el sample base
        semitonos_diff = nota['pitch'] - nota_base_midi

        # Aplicar tu algoritmo PSOLA (Inciso B)
        audio_nota = pitch_shift(audio_base, pitch_marks_base, semitonos_diff, fs)
        #audio_nota = procesar_nota_psola(
        #                                    senal = audio_base,
        #                                    fs = fs,
        #                                    pitch_marks = pitch_marks_base,
        #                                    semitonos = semitonos_diff,
        #                                    duracion_deseada_seg = nota['duracion']  # <--- ¡Esto evita el tiempo muerto!
        #                                )


        # Ajustar la duración de la nota procesada a la que pide el MIDI
        muestras_deseadas = int(nota['duracion'] * fs)
        if len(audio_nota) > muestras_deseadas:
            # Si el audio sintetizado es más largo, lo recortamos (puedes aplicarle un fade-out/envolvente aquí)
            audio_nota = audio_nota[:muestras_deseadas]
        else:
            # Si quedó más corto, lo rellenamos con ceros
            padding = np.zeros(muestras_deseadas - len(audio_nota))
            audio_nota = np.concatenate([audio_nota, padding])


        # Calcular la posición de inicio en el lienzo final (en muestras)
        indice_inicio = int(nota['inicio'] * fs)
        indice_fin = indice_inicio + len(audio_nota)

        # SUMAR el audio de la nota en el lienzo (Mezcla multi-fónica)
        audio_final[indice_inicio:indice_fin] += audio_nota

    # Normalizar el audio final para evitar que sature (clipping) antes de guardar
    if np.max(np.abs(audio_final)) > 0:
        audio_final = audio_final / np.max(np.abs(audio_final))

    return audio_final, fs

### D

In [12]:
import mido
from mido import MidiFile, MidiTrack, Message

# ==========================================
# PASO 1: GENERAR UN ARCHIVO MIDI DE PRUEBA (una sola nota)
# ==========================================
print("--- Generando archivo MIDI de prueba (una sola nota) ---")

# Crear un nuevo objeto MIDI
midi_nuevo = MidiFile()
track = MidiTrack()
midi_nuevo.tracks.append(track)

# Configurar el tempo (opcional, por defecto es 120 BPM si no se especifica)
track.append(mido.MetaMessage('set_tempo', tempo=500000, time=0))

# Definimos una sola nota (C4, MIDI note 60) para probar
nota_prueba = 60 # C4

# 1. Encender la nota
track.append(Message('note_on', note=nota_prueba, velocity=64, time=100))

# 2. Apagar la nota después de 960 ticks (equivalente a dos negras)
track.append(Message('note_off', note=nota_prueba, velocity=64, time=500))

# Definimos una sola nota (C4, MIDI note 60) para probar
nota_prueba = 64

# 1. Encender la nota
track.append(Message('note_on', note=nota_prueba, velocity=64, time=100))

# 2. Apagar la nota después de 960 ticks (equivalente a dos negras)
track.append(Message('note_off', note=nota_prueba, velocity=64, time=500))

# Definimos una sola nota (C4, MIDI note 60) para probar
nota_prueba = 60 # C4

# 1. Encender la nota
track.append(Message('note_on', note=nota_prueba, velocity=64, time=100))

# 2. Apagar la nota después de 960 ticks (equivalente a dos negras)
track.append(Message('note_off', note=nota_prueba, velocity=64, time=500))

# Definimos una sola nota (C4, MIDI note 60) para probar
nota_prueba = 67

# 1. Encender la nota
track.append(Message('note_on', note=nota_prueba, velocity=64, time=100))

# 2. Apagar la nota después de 960 ticks (equivalente a dos negras)
track.append(Message('note_off', note=nota_prueba, velocity=64, time=500))

# Definimos una sola nota (C4, MIDI note 60) para probar
nota_prueba = 65

# 1. Encender la nota
track.append(Message('note_on', note=nota_prueba, velocity=64, time=100))

# 2. Apagar la nota después de 960 ticks (equivalente a dos negras)
track.append(Message('note_off', note=nota_prueba, velocity=64, time=500))

# Definimos una sola nota (C4, MIDI note 60) para probar
nota_prueba = 62

# 1. Encender la nota
track.append(Message('note_on', note=nota_prueba, velocity=64, time=100))

# 2. Apagar la nota después de 960 ticks (equivalente a dos negras)
track.append(Message('note_off', note=nota_prueba, velocity=64, time=500))

# Guardar el archivo en el entorno de Colab
nombre_archivo_single_note = "single_note_c4.mid"
midi_nuevo.save(nombre_archivo_single_note)
print(f"¡Archivo '{nombre_archivo_single_note}' generado y guardado con éxito!\n")


# ==========================================
# PASO 2: LEER EL ARCHIVO MIDI GENERADO
# ==========================================
print("--- Leyendo el archivo MIDI generado ---")

midi_leido = MidiFile(nombre_archivo_single_note)

for i, track_leido in enumerate(midi_leido.tracks):
    print(f"Pista {i} - Total de mensajes: {len(track_leido)}")
    print("-" * 40)

    tiempo_acumulado_ticks = 0

    for msg in track_leido:
        tiempo_acumulado_ticks += msg.time

        if msg.type == 'note_on' and msg.velocity > 0:
            print(f"[Tick {tiempo_acumulado_ticks:04d}] 🎹 NOTA ENCENDIDA -> Nota MIDI: {msg.note} | Velocidad: {msg.velocity}")
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            print(f"[Tick {tiempo_acumulado_ticks:04d}] 🛑 NOTA APAGADA   -> Nota MIDI: {msg.note}")
        elif msg.is_meta:
            print(f"[Tick {tiempo_acumulado_ticks:04d}] ⚙️ MENSAJE META   -> {msg}")


--- Generando archivo MIDI de prueba (una sola nota) ---
¡Archivo 'single_note_c4.mid' generado y guardado con éxito!

--- Leyendo el archivo MIDI generado ---
Pista 0 - Total de mensajes: 14
----------------------------------------
[Tick 0000] ⚙️ MENSAJE META   -> MetaMessage('set_tempo', tempo=500000, time=0)
[Tick 0100] 🎹 NOTA ENCENDIDA -> Nota MIDI: 60 | Velocidad: 64
[Tick 0600] 🛑 NOTA APAGADA   -> Nota MIDI: 60
[Tick 0700] 🎹 NOTA ENCENDIDA -> Nota MIDI: 64 | Velocidad: 64
[Tick 1200] 🛑 NOTA APAGADA   -> Nota MIDI: 64
[Tick 1300] 🎹 NOTA ENCENDIDA -> Nota MIDI: 60 | Velocidad: 64
[Tick 1800] 🛑 NOTA APAGADA   -> Nota MIDI: 60
[Tick 1900] 🎹 NOTA ENCENDIDA -> Nota MIDI: 67 | Velocidad: 64
[Tick 2400] 🛑 NOTA APAGADA   -> Nota MIDI: 67
[Tick 2500] 🎹 NOTA ENCENDIDA -> Nota MIDI: 65 | Velocidad: 64
[Tick 3000] 🛑 NOTA APAGADA   -> Nota MIDI: 65
[Tick 3100] 🎹 NOTA ENCENDIDA -> Nota MIDI: 62 | Velocidad: 64
[Tick 3600] 🛑 NOTA APAGADA   -> Nota MIDI: 62
[Tick 3600] ⚙️ MENSAJE META   -> MetaMe

In [13]:
melodia_intro = [
    (60, 240),  # Do4 (corchea) - "O-"
    (64, 240),  # Mi4 (corchea) - "íd"
    (60, 480),  # Do4 (negra)   - "mor-"
    (67, 480),  # Sol4 (negra)  - "ta-"
    (65, 480),  # Fa4 (negra)   - "les"
    (62, 480),  # Re4 (negra)   - "el"
    (59, 240),  # Si3 (corchea) - "gri-" (Nota: es el Si debajo del Do central)
    (59, 240),  # Si3 (corchea) - "-to"
    (59, 240),  # Si3 (corchea) - "sa-"
    (59, 240),  # Si3 (corchea) - "gra-"
    (60, 960),  # Do4 (blanca)  - "do"
    (62, 480),  # Re4 (negra)   - (Adaptación para "li-ber")
    (60, 960),  # Do4 (blanca)  - "tad"
]

import mido
from mido import Message, MidiFile, MidiTrack, MetaMessage

# Crear el archivo MIDI y la pista principal
mid = MidiFile()
track = MidiTrack()
mid.tracks.append(track)

# Configurar el tempo (76 BPM según la partitura original)
# Fórmula: 60,000,000 / BPM = microsegundos por negra
track.append(MetaMessage('set_tempo', tempo=789473, time=0))

# Generación de la secuencia de notas en el archivo
tiempo_acumulado = 0

for nota, duracion in melodia_intro:
    if nota == 0:
        # Si es un silencio (nota 0), sumamos el tiempo de espera al próximo evento
        tiempo_acumulado += duracion
    else:
        # 1. Encender la nota (aplicando cualquier silencio previo en el parámetro 'time')
        track.append(Message('note_on', note=nota, velocity=85, time=tiempo_acumulado))

        # 2. Apagar la nota después de la duración correspondiente
        track.append(Message('note_off', note=nota, velocity=85, time=duracion))

        # Reiniciar el acumulador de tiempo ya que se acaba de tocar una nota
        tiempo_acumulado = 0

# Guardar el resultado en un archivo
mid.save('intro_himno_argentino.mid')
print("Archivo 'intro_himno_argentino.mid' creado con éxito.")

Archivo 'intro_himno_argentino.mid' creado con éxito.


In [19]:
# Rutas de ejemplo (ajústalas a tus carpetas de Drive)
archivo_midi_single_note = "single_note_c4.mid" # Usamos el nuevo archivo MIDI de una sola nota
sample_guitarra_C4 = "Guitar.ff.sulA.C4E4.aif"

# Sintetizar la obra completo (ahora solo una nota)
audio_sintetizado, fs = sintetizar_midi_a_audio(archivo_midi_single_note, sample_guitarra_C4, nota_base_midi=60)

# Guardar el resultado en formato .wav para la entrega
sf.write("Resultado_single_note_c4.wav", audio_sintetizado, fs) # Nuevo nombre de archivo para el resultado
print("¡Síntesis de nota única completada con éxito!")


LibsndfileError: Error opening 'Guitar.ff.sulA.C4E4.aif': System error.

In [20]:
import soundfile as sf
import numpy as np

import numpy as np
import matplotlib.pyplot as plt

def analizar_artefactos(audio_proc, fs, pitch_marks_proc=None):
    """
    Genera gráficos orientados a descubrir discontinuidades temporales (clics)
    o efectos metálicos (filtrado en peine por mala fase).
    """

    f0_base_detectado, periodo_base_detectado = estimar_f0_autocorrelacion(audio_proc, current_fs)
    print(f"Frecuencia fundamental detectada en el sample base 'Guitar.ff.sulA.C4E4.aif': {f0_base_detectado:.2f} Hz")
    pitch_marks_base = detectar_pitch_marks(audio_proc, periodo_base_detectado)

    plt.figure(figsize=(14, 8))

    # 1. ZOOM TEMPORAL: Buscar discontinuidades / saltos abruptos (clics)
    # Graficamos una ventana muy corta de tiempo (ej. 40 milisegundos)
    tiempo = np.arange(len(audio_proc)) / fs

    # Tomamos un segmento intermedio del audio para inspeccionar las uniones de ventanas
    idx_inicio = int(len(audio_proc) * 0.0)
    idx_fin = int(len(audio_proc) * 1.0)

    plt.plot(tiempo[idx_inicio:idx_fin] * 1000, audio_proc[idx_inicio:idx_fin], color='crimson', lw=1.5)
    plt.plot(tiempo[pitch_marks_base]*1000, audio_proc[pitch_marks_base], ls=' ', marker='x')
    plt.title("Zoom Temporal del Sample de Guitarra C4")
    plt.xlabel("Tiempo [ms]")
    plt.ylabel("Amplitud")
    plt.grid(True)
    plt.xlim(0, 200)
    plt.show()

# Re-usamos la ruta del sample de guitarra C4 y la función estimar_f0_autocorrelacion
# (que se definió en la celda dACvww8oyASA)

print("--- Verificando la frecuencia fundamental del sample base (Guitar.ff.sulA.C4E4.aif) ---")

try:
    # Cargar el sample base .aif (en Mono)
    audio_base_original, fs_original = sf.read(sample_guitarra_C4)
    if len(audio_base_original.shape) > 1:
        audio_base_original = np.mean(audio_base_original, axis=1)

    # recorto solo una nota
    audio_base_original = audio_base_original[int(fs_original*0.075):int(fs_original*3)]

    # Asegurarse de que fs coincide con la global si es posible
    if 'fs' not in globals() or fs_original != fs:
        print(f"Advertencia: la frecuencia de muestreo del sample base ({fs_original} Hz) no coincide con la global ({fs} Hz). Se usará fs_original para este análisis.")
        current_fs = fs_original
    else:
        current_fs = fs


    analizar_artefactos(audio_base_original, fs_original)

    # Reproducir el audio original para verificar
    from IPython.display import Audio, display
    print(f"Reproduciendo el sample de audio original: {sample_guitarra_C4}")
    display(Audio(audio_base_original, rate=fs_original))

except FileNotFoundError:
    print(f"ERROR: Archivo no encontrado en {sample_guitarra_C4}. Por favor, verifica la ruta y asegúrate de que el archivo exista.")
except Exception as e:
    print(f"Ocurrió un error al cargar o procesar el audio base: {e}")


--- Verificando la frecuencia fundamental del sample base (Guitar.ff.sulA.C4E4.aif) ---
Ocurrió un error al cargar o procesar el audio base: Error opening 'Guitar.ff.sulA.C4E4.aif': System error.


In [21]:
def analizar_artefactos(audio_proc, fs, pitch_marks_proc=None):
    """
    Genera gráficos orientados a descubrir discontinuidades temporales (clics)
    o efectos metálicos (filtrado en peine por mala fase).
    """
    plt.figure(figsize=(14, 8))

    # 1. ZOOM TEMPORAL: Buscar discontinuidades / saltos abruptos (clics)
    # Graficamos una ventana muy corta de tiempo (ej. 40 milisegundos)
    tiempo = np.arange(len(audio_proc)) / fs

    # Tomamos un segmento intermedio del audio para inspeccionar las uniones de ventanas
    idx_inicio = int(len(audio_proc) * 0.0)
    idx_fin = int(len(audio_proc) * 1.0)

    plt.plot(tiempo[idx_inicio:idx_fin] * 1000, audio_proc[idx_inicio:idx_fin], color='crimson', lw=1.5)
    plt.title("Zoom Temporal Audio Sintetizado PSOA")
    plt.xlabel("Tiempo [ms]")
    plt.ylabel("Amplitud")
    plt.grid(True)
    plt.xlim(1800, 2000) # discontinuidad
    #plt.xlim(0, 4000)
    plt.show()



analizar_artefactos(audio_sintetizado, fs)
from IPython.display import Audio, display
print(f"Reproduciendo el Audio Sintetizado")
display(Audio(audio_sintetizado, rate=fs))

NameError: name 'audio_sintetizado' is not defined

In [ ]:
from scipy.signal import welch

freqs_synth, psd_synth = welch(audio_sintetizado, fs=fs, nperseg=2**16)
freqs_original, psd_original = welch(audio_base_original, fs=fs_original, nperseg=2**16)

plt.figure(figsize=(10, 5))
plt.semilogy(freqs_original, psd_original, label="Original audio")
plt.semilogy(freqs_synth, psd_synth, label="Synth")
plt.title("Welch PSD of Original Audio and Synth")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density")

plt.grid(True, which="both", linestyle="--", alpha=0.3)
plt.tight_layout()
#plt.xlim([40, 60])
plt.xlim([0, 750])
#plt.ylim([1e-11, 1e-1])
plt.axvline(x=261.63, ls='--', c='tab:green', alpha=0.5, label='Do')
plt.axvline(x=293.66, ls='--', c='tab:red', alpha=0.5, label='Re')
plt.axvline(x=329.63, ls='--', c='tab:purple', alpha=0.5, label='Mi')
plt.axvline(x=349.23, ls='--', c='tab:brown', alpha=0.5, label='Fa')
plt.axvline(x=392.00, ls='--', c='tab:pink', alpha=0.5, label='Sol')

plt.legend()
plt.show()

# 2.2 Karplus Strong

## Modelo original

In [ ]:
#Construyo funcion transferencia
def karplusStrongOriginalTF(L, RL):
  bH = [0.5, 0.5]
  aH = []
  aH.append(1)
  for zb in range(L-1):
      aH.append(0)
  aH.append(-0.5*RL)
  aH.append(-0.5*RL)
  return bH,aH

bH,aH = karplusStrongOriginalTF(100,0.97)
#Grafico respuesta en frecuencia
w,h = sp.freqz(bH,aH)
plt.figure(figsize=(10, 5))
plt.semilogx(w, 20*np.log10(abs(h)), label="L=10 y RL=0.75")
plt.legend()
plt.xlabel("Frecuencia(Hz)")
plt.ylabel("Magnitud(dB)")
plt.title("Respuesta en frecuencia para L=10 y RL=0.75")
plt.grid(True)
plt.show()
plt.figure(figsize=(10, 5))
plt.semilogx(w, np.angle(h, deg=True), label="L=10 y RL=0.75")
plt.legend()
plt.xlabel("Frecuencia(Hz)")
plt.ylabel("Fase(°)")
plt.title("Respuesta en frecuencia para L=10 y RL=0.75")
plt.grid(True)
plt.show()
# Saco y grafico polos y ceros
zeros = np.roots(bH)
poles = np.roots(aH)
theta = np.linspace(0, 2*np.pi, 1000)
plt.figure(figsize=(6, 6))
plt.plot(np.cos(theta), np.sin(theta), 'k--', lw=0.8)
plt.plot(zeros.real, zeros.imag, 'go', ms=9, label='Ceros')
plt.plot(poles.real, poles.imag, 'rx', ms=9, mew=2, label='Polos')
plt.axhline(0, color='k', lw=0.5)
plt.axvline(0, color='k', lw=0.5)
plt.axis('equal')
plt.legend()
plt.grid(True, alpha=0.3)
plt.title("Polos y ceros para L=10 y RL=0.75")
plt.xlabel("Parte real")
plt.ylabel("Parte imaginaria")
plt.show()

## Implementación

### Ruido uniforme

In [ ]:
from IPython.display import Audio, display

def Lfromfrec(f0):
  return int(np.round((fs/f0) - (1/2)))

#Genero muestra finita de ruido blanco
soundsec = 1
noisesec = 0.01
noise = np.random.rand(int(noisesec*fs))
#La entrada es ese cacho de ruido mas el resto de silencio
inputsignal = np.concatenate((noise, np.zeros(int(fs*(soundsec-noisesec))))) # Concatenar en un solo array
display(Audio(inputsignal, rate=fs))

b1,a1 = karplusStrongOriginalTF(Lfromfrec(130.81),0.99) #Do
b2,a2 = karplusStrongOriginalTF(Lfromfrec(146.83),0.95) #Re
b3,a3 = karplusStrongOriginalTF(Lfromfrec(164.81),0.66) #Mi

out1 = sp.lfilter(b1,a1,inputsignal)
out2 = sp.lfilter(b2,a2,inputsignal)
out3 = sp.lfilter(b3,a3,inputsignal)

#Reproduzco las muestras

display(Audio(out1, rate=fs))
display(Audio(out2, rate=fs))
display(Audio(out3, rate=fs))

### Ruido normal

In [ ]:
#Genero muestra finita de ruido normal
soundsec = 1
noisesec = 0.01
noise = np.random.randn(int(noisesec*fs))
#La entrada es ese cacho de ruido mas el resto de silencio
inputsignalnorm = np.concatenate((noise, np.zeros(int(fs*(soundsec-noisesec))))) # Concatenar en un solo array
display(Audio(inputsignalnorm, rate=fs))

b1,a1 = karplusStrongOriginalTF(Lfromfrec(130.81),0.99) #Do
b2,a2 = karplusStrongOriginalTF(Lfromfrec(146.83),0.95) #Re
b3,a3 = karplusStrongOriginalTF(Lfromfrec(164.81),0.66) #Mi

out1norm = sp.lfilter(b1,a1,inputsignalnorm)
out2norm = sp.lfilter(b2,a2,inputsignalnorm)
out3norm = sp.lfilter(b3,a3,inputsignalnorm)

#Normalizo
out1norm = out1norm/np.max(np.abs(out1norm))
out2norm = out2norm/np.max(np.abs(out2norm))
out3norm = out3norm/np.max(np.abs(out3norm))

#Reproduzco las muestras

display(Audio(out1norm, rate=fs))
display(Audio(out2norm, rate=fs))
display(Audio(out3norm, rate=fs))

### Mejora

In [ ]:
# Importo la respuesta al impulso de una guitarra en particular(esta en la carpeta del jupyter notes)
# Vas a archivo->ubicar en Drive->descargas el.wav->vas a la parte de archivos en la barra izquierda->subis el archivo
import soundfile as sf
from math import gcd

ir, ir_fs = sf.read("IR_Martin 017 Schatten HFN_bip44100_CSM.wav")
# If the IR is stereo, just take one channel
if ir.ndim > 1:
    ir = ir[:, 0]
# Resampleo
g = gcd(fs, ir_fs)
ir = sp.resample_poly(ir, fs//g, ir_fs//g)

#uso las de ruido blanco porque capaz se nota mas la mejora igual yo no noto nada
out1guitar = sp.convolve(out1, ir)
out2guitar = sp.convolve(out2, ir)
out3guitar = sp.convolve(out3, ir)

# Normalizo
out1guitar = out1/np.max(np.abs(out1guitar))
out2guitar = out2/np.max(np.abs(out2guitar))
out3guitar = out3/np.max(np.abs(out3guitar))

#Reproduzco las muestras

display(Audio(out1guitar, rate=fs))
display(Audio(out2guitar, rate=fs))
display(Audio(out3guitar, rate=fs))


### L racional

In [ ]:
# https://en.wikipedia.org/wiki/Karplus%E2%80%93Strong_string_synthesis

#Aca no uso una funcion separada para sacar L
def karplusStrongFractionalTF(f0, fs, RL):
  L_exact = fs / f0 - 0.5
  #Divido entre parte entera y fraccionaria, despues devuelvo la fraccionaria para hacer interpolacion
  M = int(np.floor(L_exact))
  eta = L_exact - M
  #Construyo funcion transferencia solo con parte entera
  bH = [0.5, 0.5]
  aH = [1] + [0]*(M-1) + [-0.5*RL, -0.5*RL]

  return bH, aH, eta

b1,a1,eta = karplusStrongFractionalTF(130.81,fs,0.99) #Do
outEntero = sp.lfilter(b1,a1,inputsignalnorm) #Uso ruido normal
outInterpol = sp.lfilter([1-eta, eta], [1], outEntero) #Interpolación lineal entre muestras

display(Audio(outInterpol, rate=fs))


## Modelo con b

In [ ]:
#AHORA TENGO QUE HACER OTRA FUNCION PORQUE EL COEFICIENTE ESTE CAMBIA CADA SAMPLE NOOOOOO
#Aprovecho y hago un módulo completo que no dependa de los anteriores y se puede poner en la aplicación final

def karplusStrong(f0, fs, RL, b, duration=1.0, noisedur=0.1, noisetype='normal', boxResonance=False):
  L_exact = fs / f0 - 0.5
  #Divido entre parte entera y fraccionaria, despues devuelvo la fraccionaria para hacer interpolacion
  M = int(np.floor(L_exact))
  eta = L_exact - M

  #Genero señal x
  N = int(duration*fs)
  if(noisetype == 'blanco'):
    noise = np.random.rand(int(noisedur * fs))
  elif (noisetype == 'normal'):
    noise = np.random.randn(int(noisedur * fs))
  else:
    raise ValueError("noisetype debe ser 'blanco' o 'normal'")
  x = np.concatenate((noise, np.zeros(N - int(noisedur * fs))))
  #sé que no es la implementación mas eficiente esta pero funcionar funciona.
  xA = np.zeros(N)

  #Lo paso por el sistema
  y = np.zeros(N)
  for n in range(N):
    xA[n] = x[n]+RL*y[n-M] if n >= M else x[n]
    sign = 1 if np.random.rand() < b else -1
    y[n] = sign*(0.5*xA[n]+0.5*xA[n-1] if n>0 else 0.5*xA[n])

  # Interpolación lineal para la parte racional de L
  y = sp.lfilter([1-eta, eta], [1.0], y)

  if(boxResonance):
    #si no esta el archivo wav no se crashea todo el código, solo printea un error y listo
    try:
      ir, ir_fs = sf.read("IR_Martin 017 Schatten HFN_bip44100_CSM.wav")
      if ir.ndim > 1:
          ir = ir[:, 0]
      # Resampleo
      g = gcd(fs, ir_fs)
      ir = sp.resample_poly(ir, fs//g, ir_fs//g)
      y = sp.convolve(y, ir)
    except (sf.LibsndfileError):
      print("ERROR: Archivo .wav no está en la carpeta del proyecto.")

  return y / np.max(np.abs(y))

outDo = karplusStrong(130.81,fs,0.995,1,1.0,0.05,"normal", True) #Do
outRe = karplusStrong(146.83,fs,0.99,0.95,1.0,0.05,"normal") #Do
outMi = karplusStrong(164.81,fs,0.5,0.01,1.0,0.05,"normal") #Do

display(Audio(outDo, rate=fs))
display(Audio(outRe, rate=fs))
display(Audio(outMi, rate=fs))
#Suena horrible para cualquier valor de b que no este muy cerca ni de 0 ni de 1. 0.5 suena como puro ruido. Creo que eso esta bien igual

Esta parte ya está. El punto H es teórico porque hay que hacer la función transferencia dejando al coeficiente de b como su promedio, E[p]=1.b +(1-b).(-1)=2b-1 y ahi analizar la fase. El punto i no lo entiendo bien, pero pide encontrar una expresión, no implementarlo.

## 2.3 Sintesis aditiva

In [18]:
import soundfile as sf
from scipy.fft import fft, fftfreq
from IPython.display import Audio, display # Moved here for immediate use

# --- CONFIGURACIÓN: Ajusta esta ruta a tu sample de audio ---
# Originalmente se cargaba desde un archivo, ahora usaremos el audio sintetizado de MIDI
# Asegúrate de que las celdas de síntesis MIDI (como VSXv9gy12nIE) hayan sido ejecutadas.

# Usamos el audio y fs generados por la síntesis MIDI
# audio_sintetizado y fs deben estar disponibles en el entorno global.
# Si esta celda se ejecuta antes de la síntesis MIDI, audio y fs no estarán definidos.

if 'audio_sintetizado' in locals() and 'fs' in locals():
    audio = audio_sintetizado
    # fs ya está definida en el entorno global por la síntesis MIDI
    print(f"Audio para síntesis aditiva tomado de la síntesis MIDI. Muestras: {len(audio)}, Fs: {fs} Hz")
else:
    print("ERROR: 'audio_sintetizado' o 'fs' no están definidos. Por favor, ejecuta las celdas de síntesis MIDI primero.")
    audio = None # Para evitar errores posteriores si el audio no se carga

# Reproducir el audio cargado para verificar
if audio is not None:
    print(f"Reproduciendo el audio (sintetizado de MIDI):")
    display(Audio(audio, rate=fs))


ERROR: 'audio_sintetizado' o 'fs' no están definidos. Por favor, ejecuta las celdas de síntesis MIDI primero.


Ahora, vamos a calcular la FFT de un segmento estacionario del audio. Elegiremos una ventana en la mitad del sample para asegurar que estamos en la parte sostenida de la nota. La elección del tamaño de la ventana es crucial para el análisis espectral.

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from IPython.display import Audio, display

# 1. VERIFICACIÓN: Traemos el audio de la síntesis MIDI
variable_audio_midi = 'audio_sintetizado'

if variable_audio_midi in globals() or variable_audio_midi in locals():
    # Buscamos la variable dinámicamente en el entorno
    audio = globals().get(variable_audio_midi, locals().get(variable_audio_midi))
    print(f"✅ ¡Conectado! Audio tomado de la síntesis MIDI.")
    print(f"Muestras totales: {len(audio)} | Frecuencia de muestreo (fs): {fs} Hz")

    # Si es estéreo, lo pasamos a mono para la FFT
    if audio.ndim > 1:
        audio = audio[:, 0]
else:
    print("❌ ERROR: No encontré 'audio_sintetizado'.")
    print("Por favor, fijate cómo se llama la variable de salida en tu celda de síntesis MIDI y poné ese nombre arriba.")
    audio = None

# 2. PROCESAMIENTO Y FFT (Solo si el audio existe)

if audio is not None:
    # 1. Quitar el valor medio (DC offset)
    audio_limpio = audio - np.mean(audio)

    # 2. Encontrar la parte con más volumen (energía) del audio para no analizar silencio
    # Buscamos dónde está el pico máximo de la onda
    indice_maximo = np.argmax(np.abs(audio_limpio))

    N_fft = 16384
    # Nos paramos un poquito antes del pico máximo para agarrar el cuerpo de la nota
    inicio_analisis = max(0, indice_maximo - 500)

    # Verificamos que no nos salgamos del final del audio
    if inicio_analisis + N_fft > len(audio_limpio):
        # Si el audio es muy corto, usamos lo que haya rellenando con ceros (zero-padding)
        segmento = audio_limpio[inicio_analisis:]
    else:
        segmento = audio_limpio[inicio_analisis : inicio_analisis + N_fft]

    print(f"Analizando un segmento de {len(segmento)} muestras alrededor de la zona con más energía.")

    # 3. Aplicar ventana de Hann
    segmento_ventanado = segmento * np.hanning(len(segmento))

    # 4. Calcular FFT
    X = fft(segmento_ventanado, n=N_fft) # Forzamos N_fft puntos
    magnitud = np.abs(X)[:N_fft//2]
    frecuencias = fftfreq(N_fft, 1/fs)[:N_fft//2]

    # 5. Graficar con escala Logarítmica si sigue viéndose muy chico
    plt.figure(figsize=(10, 4))
    plt.plot(frecuencias, magnitud, color='firebrick', label='Espectro Optimizado')
    plt.xlim(0, 3000)  # Rango audible de armónicos de la nota
    plt.xlabel("Frecuencia [Hz]")
    plt.ylabel("Magnitud")
    plt.grid(True, linestyle=':')
    plt.legend()
    plt.savefig("analisis_espectro_fft.pdf", format='pdf', dpi=300, bbox_inches='tight')

    plt.show()

    # Mostrar el reproductor de ese pedacito exacto para auditar
    print("🔊 Esto es exactamente lo que la FFT está analizando:")
    display(Audio(segmento, rate=fs))

❌ ERROR: No encontré 'audio_sintetizado'.
Por favor, fijate cómo se llama la variable de salida en tu celda de síntesis MIDI y poné ese nombre arriba.


In [17]:
import numpy as np
from scipy.signal import find_peaks

# =====================================================================
# CONFIGURACIÓN GENÉRICA: Esto te lo da el evento MIDI en cada momento
# =====================================================================
# Supongamos que el MIDI te pidió una nota cuya frecuencia teórica es esta:
f_midi_teorica = 330.0  # Cambia automáticamente según la nota del MIDI (ej: 330Hz para Mi)

# =====================================================================
# ALGORITMO ROBUSTO DE EXTRACCIÓN DE PARCIALES
# =====================================================================
fk_finales = []
Ak_finales = []

# Margen de tolerancia para buscar el pico real alrededor del múltiplo teórico
# Usamos un porcentaje de la fundamental para que sea adaptativo
tolerancia = f_midi_teorica * 0.1  # 10% de margen

K_armonicos = 5  # Cantidad de parciales que querés extraer

for k in range(1, K_armonicos + 1):
    f_armonico_teorica = k * f_midi_teorica

    # Creamos una ventana/máscara alrededor de la frecuencia del armónico k
    indices_zona = np.where(
        (frecuencias >= f_armonico_teorica - tolerancia) &
        (frecuencias <= f_armonico_teorica + tolerancia)
    )[0]

    if len(indices_zona) > 0:
        # Buscamos el índice del valor MÁXIMO de magnitud en esa zona específica
        idx_max_zona = np.argmax(magnitud[indices_zona])
        idx_pico_real = indices_zona[idx_max_zona]

        # Guardamos la frecuencia y la amplitud real detectada por la FFT
        fk_finales.append(frecuencias[idx_pico_real])
        Ak_finales.append(magnitud[idx_pico_real])
    else:
        # Si por alguna razón no encuentra el armónico, dejamos el valor teórico
        fk_finales.append(f_armonico_teorica)
        Ak_finales.append(0.0)

# =====================================================================
# NORMALIZACIÓN RESPECTO AL PRIMER ARMÓNICO REAL
# =====================================================================
fk_finales = np.array(fk_finales)
Ak_finales = np.array(Ak_finales)

A1 = Ak_finales[0] if Ak_finales[0] > 0 else 1.0
Ak_normalizadas = Ak_finales / A1

# =====================================================================
# MOSTRAR LA TABLA GENÉRICA
# =====================================================================
print("==========================================================")
print("     EXTRACCIÓN ADAPTATIVA DE COMPONENTES                 ")
print("==========================================================")
print(f"🎵 Frecuencia Guía del MIDI: {f_midi_teorica:.2f} Hz")
print(f"🎵 Fundamental Real Detectada (f0): {fk_finales[0]:.2f} Hz\n")

print("Parcial (k)  |  Frecuencia Real (fk)  |  Amplitud Relativa (Ak)")
print("----------------------------------------------------------")
for i in range(K_armonicos):
    print(f" Armónico {i+1}  |      {fk_finales[i]:7.2f} Hz      |       {Ak_normalizadas[i]:.4f}")
print("==========================================================")

NameError: name 'frecuencias' is not defined

In [22]:
import numpy as np
from IPython.display import Audio, display

def sintesis_aditiva_sin_envolvente(fk_lista, Ak_norm_lista, duracion, fs=44100):
    """
    Sintetiza una nota sumando componentes senoidales puras (Parciales).

    fk_lista: Array o lista con las frecuencias reales extraídas (f1, f2, f3...)
    Ak_norm_lista: Array o lista con las amplitudes relativas (normalizadas respecto a f1)
    duracion: Tiempo en segundos que va a durar la nota (ej: 2.0)
    fs: Frecuencia de muestreo (por defecto 44100 Hz, estándar de audio)
    """
    # 1. Crear el vector de tiempo t
    t = np.arange(0, duracion, 1/fs)

    # 2. Inicializar el vector de audio en cero
    audio_sintetizado = np.zeros_like(t)

    # 3. Aplicar la sumatoria matemática: x(t) = SUM( Ak * sin(2 * pi * fk * t) )
    for fk, Ak_norm in zip(fk_lista, Ak_norm_lista):
        # Sumamos cada armónico puro
        audio_sintetizado += Ak_norm * np.sin(2 * np.pi * fk * t)

    # 4. NORMALIZACIÓN CRÍTICA (Evita la saturación/clipping)
    # Al sumar muchas ondas, la amplitud puede pasar de 1.0 y romper los parlantes.
    # Llevamos el pico máximo del audio a un rango seguro entre -1.0 y 1.0.
    if np.max(np.abs(audio_sintetizado)) > 0:
        audio_sintetizado /= np.max(np.abs(audio_sintetizado))

    return audio_sintetizado, t


In [23]:

# =====================================================================
# EJECUCIÓN: Generamos la nota musical sintética
# =====================================================================
duracion_nota = 1.0  # Duración en segundos (podés cambiarlo a gusto)

# Llamamos a nuestra función pasándole la data que guardamos en la celda anterior
audio_aditivo, vector_tiempo = sintesis_aditiva_sin_envolvente(
    fk_finales,
    Ak_normalizadas,
    duracion=duracion_nota,
    fs=fs
)

print("==========================================================")
print("     ¡SÍNTESIS ADITIVA COMPLETADA CON ÉXITO!              ")
print("==========================================================")
print(f"Nota generada a partir de {len(fk_finales)} armónicos.")
print(f"Duración: {duracion_nota} segundos | Frecuencia de muestreo: {fs} Hz")
print("==========================================================")

# Reproducir el resultado matemático
print("\n🔊 Escuchá tu instrumento creado por Síntesis Aditiva:")
display(Audio(audio_aditivo, rate=fs))

NameError: name 'Ak_normalizadas' is not defined

### Recuperación rectificador + LPF

Ahora sacamos la envolvente de la señal original para hacer el ADSR. En primer lugar probamos con una rectificación (el audio tiene media nula) + filtro pasabajos para ver como varia dinámicamente la amplitud de la señal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter

# =====================================================================
# EXTRAER LA ENVOLVENTE DESDE EL AUDIO MIDI SINTETIZADO
# =====================================================================

def extraer_envolvente_eficiente(signal, fs, cutoff_hz=10.0):
    # 1. Rectificación (todo a positivo)
    senal_rectificada = np.abs(signal)

    # 2. Filtro pasa-bajos
    wc = 2 * np.pi * cutoff_hz / fs
    b = [wc / (1 + wc)]
    a = [1, -1 / (1 + wc)]

    # Aplicamos el filtro
    envolvente = lfilter(b, a, senal_rectificada)
    return envolvente

# --- Parámetros ---
frecuencia_corte = 15.0

# Extraemos la envolvente completa primero
envolvente_extraida = extraer_envolvente_eficiente(audio_sintetizado, fs, cutoff_hz=frecuencia_corte)

# =====================================================================
# RECORTE DE LA SEÑAL NO NULA
# =====================================================================
# Definimos un umbral para considerar qué es "silencio" (ajustable si es necesario)
umbral_silencio = 1e-4

# Buscamos los índices donde la envolvente supera ese umbral
indices_con_senal = np.where(envolvente_extraida > umbral_silencio)[0]

if len(indices_con_senal) > 0:
    # El último índice donde hubo sonido relevante
    ultimo_indice_valido = indices_con_senal[-1]

    # Recortamos los arrays para quedarnos solo con la parte activa
    audio_recortado = audio_sintetizado[:ultimo_indice_valido]
    envolvente_recortada = envolvente_extraida[:ultimo_indice_valido]
else:
    # Si por alguna razón toda la señal es plana, no recortamos nada para evitar errores
    audio_recortado = audio_sintetizado
    envolvente_recortada = envolvente_extraida
    ultimo_indice_valido = len(audio_sintetizado)

# Creamos el vector de tiempo basado únicamente en la duración del segmento recortado
tiempo_recortado = np.linspace(0, len(audio_recortado) / fs, len(audio_recortado))
duracion_recortada = len(audio_recortado) / fs

# =====================================================================
# VISUALIZACIÓN DE RESULTADOS
# =====================================================================
plt.figure(figsize=(12, 5))

# Graficamos únicamente las versiones recortadas
plt.plot(tiempo_recortado, audio_recortado, color='silver', label='Audio MIDI Sintetizado (Activo)', alpha=0.7)
plt.plot(tiempo_recortado, envolvente_recortada, color='darkorange', linewidth=2, label=f'Envolvente Extraída ({frecuencia_corte} Hz LPF)')

plt.xlabel('Tiempo (segundos)', fontsize=12)
plt.ylabel('Amplitud', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)

# Ajustamos el límite horizontal al nuevo tiempo recortado
plt.xlim(0, duracion_recortada)

plt.tight_layout()
plt.show()

print("==========================================================")
print("     ¡EXTRACCIÓN Y RECORTE DE AUDIO COMPLETADOS!          ")
print("==========================================================")
print(f"Se eliminó el silencio final.")
print(f"Muestras originales: {len(audio_sintetizado)} -> Muestras activas: {len(audio_recortado)}")
print(f"Duración original: {len(audio_sintetizado)/fs:.2f} s -> Nueva duración: {duracion_recortada:.2f} segundos.")
print("==========================================================")

### Recuperación via trasnf de Hilbert

Ahora lo hacemos con otro método usando la transformada de Hilbert

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt

# =====================================================================
# 1. RECORTE PREVIO DEL SILENCIO
# =====================================================================
umbral_audio_real = 1e-4
indices_activos = np.where(np.abs(audio_sintetizado) > umbral_audio_real)[0]

if len(indices_activos) > 0:
    ultimo_indice = indices_activos[-1]
    muestras_margen = int(0.05 * fs)
    corte_final = min(len(audio_sintetizado), ultimo_indice + muestras_margen)
    audio_recortado = audio_sintetizado[:corte_final]
else:
    audio_recortado = audio_sintetizado

# =====================================================================
# 2. CALCULAR HILBERT (Amplitud instantánea ruidosa)
# =====================================================================
senal_analitica = hilbert(audio_recortado)
envolvente_raw = np.abs(senal_analitica)

# =====================================================================
# 3. FILTRADO DE SUAVIZADO (Remover los batidos de ~330 Hz)
# =====================================================================
# Diseñamos un filtro Butterworth pasa-bajos de orden 2.
# Cortamos a 15 Hz para dejar pasar solo los cambios lentos de volumen.
frecuencia_corte_suave = 15.0
b, a = butter(2, frecuencia_corte_suave / (0.5 * fs), btype='low')

# 'filtfilt' aplica el filtro en ambos sentidos -> CERO DESFASE TEMPORAL
envolvente_suave = filtfilt(b, a, envolvente_raw)

# Por seguridad matemática (filtfilt puede dejar pequeños valores negativos en los bordes)
envolvente_suave = np.maximum(envolvente_suave, 0)

# =====================================================================
# 4. VISUALIZACIÓN DE RESULTADOS
# =====================================================================
tiempo_recortado = np.linspace(0, len(audio_recortado) / fs, len(audio_recortado))

plt.figure(figsize=(12, 5))
plt.plot(tiempo_recortado, audio_recortado, color='silver', label='Audio MIDI Sintetizado', alpha=0.7)

# Graficamos la envolvente original de Hilbert de fondo (fina) para comparar
plt.plot(tiempo_recortado, envolvente_raw, color='navajowhite', linewidth=1, label='Hilbert Raw (Con batidos armónicos)', alpha=0.5)

# Graficamos la envolvente suavizada final (gruesa)
plt.plot(tiempo_recortado, envolvente_suave, color='darkorange', linewidth=2.5, label=f'Envolvente Hilbert + LPF ({frecuencia_corte_suave} Hz)')

plt.xlabel('Tiempo (segundos)', fontsize=12)
plt.ylabel('Amplitud', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)
plt.xlim(0, len(audio_recortado) / fs)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt

# # =====================================================================
# 0. RECORTE AGRESIVO DE SILENCIOS (Principio y Final)
# =====================================================================
# Calculamos la señal en valor absoluto para evaluar la amplitud analógica
amplitud_absoluta = np.abs(audio_sintetizado)

# Buscamos el valor máximo real de la nota
max_amplitud = np.max(amplitud_absoluta)

# Ponemos un umbral más bajo (0.005) respecto al pico máximo para ignorar el ruido inicial
umbral = 0.005 * max_amplitud

# Encontramos absolutamente TODOS los índices que superan ese umbral
indices_con_senal = np.where(amplitud_absoluta > umbral)[0]

if len(indices_con_senal) > 0:
    # El primer índice donde empieza la nota de verdad
    primer_indice = indices_con_senal[0]

    # El ÚLTIMO índice donde la nota cae por debajo del umbral definitivamente
    ultimo_indice = indices_con_senal[-1]

    # Margen de seguridad (ej: 500 muestras antes y después para no cortar transitorios)
    margen = 500
    inicio = max(0, primer_indice - margen)
    fin = min(len(audio_sintetizado), ultimo_indice + margen)

    # Recortamos el audio de punta a punta
    audio_recortado = audio_sintetizado[inicio:fin]
else:
    audio_recortado = audio_sintetizado

# A partir de acá, todo el resto de tu código (Hilbert, filtro y gráfico) sigue igual...

# =====================================================================
# 1. PASO PREVIO: CALCULAR LA SEÑAL ANALÍTICA DE HILBERT
# =====================================================================
z_completa = hilbert(audio_recortado)

# =====================================================================
# 2. TU PROPUESTA: FILTRAR ANTES DEL MÓDULO (Aislar el primer armónico)
# =====================================================================
f_fundamental = 330.0
ancho_banda = 80.0

f_inferior = f_fundamental - (ancho_banda / 2)
f_superior = f_fundamental + (ancho_banda / 2)

b_bpf, a_bpf = butter(2, [f_inferior / (0.5 * fs), f_superior / (0.5 * fs)], btype='band')
z_filtrada = filtfilt(b_bpf, a_bpf, z_completa)

# =====================================================================
# 3. CÁLCULO DEL MÓDULO (VALOR ABSOLUTO)
# =====================================================================
envolvente_pre_filtrada = np.abs(z_filtrada)
envolvente_raw = np.abs(z_completa)

# =====================================================================
# 4. VISUALIZACIÓN Y COMPARACIÓN (Con el tiempo recalculado)
# =====================================================================
# El vector de tiempo ahora se calcula en base a la longitud del audio recortado
tiempo = np.linspace(0, len(audio_recortado) / fs, len(audio_recortado))

plt.figure(figsize=(12, 5))
plt.plot(tiempo, audio_recortado, color='silver', label='Audio MIDI Recortado (Sin silencio)', alpha=0.6)
plt.plot(tiempo, envolvente_raw, color='red', linewidth=1, label='Hilbert Estándar (Sin silencio)', alpha=0.4)
plt.plot(tiempo, envolvente_pre_filtrada, color='blue', linewidth=2.5, label='Filtrar Z(t) antes del módulo (BPF a 330 Hz)')

plt.xlabel('Tiempo (segundos)', fontsize=12)
plt.ylabel('Amplitud', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', fontsize=10)
plt.xlim(0, len(audio_recortado) / fs)
plt.tight_layout()
plt.show()

## Caracterización de envolvente

In [ ]:
import numpy as np

def aplicar_envolvente_lineal_esquema(audio_sintetizado, fs, parametros_adsr, duracion_nota):
    """
    Aplica la envolvente lineal por tramos
    - Attack: rampa ascendente hasta k * A0.
    - Decay: rampa descendente hasta el inicio del Sustain.
    - Sustain: rampa descendente con pendiente (fall rate alpha) hasta A0 en el Note Off.
    - Release: caída rápida desde A0 a 0.
    """
    # 1. Tiempos del análisis original
    t_attack = parametros_adsr["tiempo_attack"]
    t_decay = parametros_adsr["tiempo_decay"]
    t_release = parametros_adsr["tiempo_release"]

    # 2. Amplitudes del esquema
    amp_max = 1.0                              # k * A0 (Pico normalizado)
    amp_sustain_final = parametros_adsr["nivel_sustain"] # A0 (Nivel nominal al Note Off)

    # Para que el Sustain tenga caída (fall rate), el inicio del sustain
    # tiene que estar más arriba que el final (A0). Lo estimamos del Note Off original.
    amp_sustain_inicio = parametros_adsr["nivel_note_off"]
    if amp_sustain_inicio <= amp_sustain_final:
        # Resguardo matemático si los niveles dieron muy pegados
        amp_sustain_inicio = amp_sustain_final * 1.2 if amp_sustain_final * 1.2 < amp_max else amp_max

    # 3. Hitos temporales acoplados a la nueva duración
    t_note_on = 0.0
    t_fin_attack = t_attack
    t_fin_decay = t_attack + t_decay
    t_note_off = duracion_nota - t_release
    t_fin_sonido = duracion_nota

    # Control de seguridad por si la nota es muy corta
    if t_fin_decay >= t_note_off:
        t_fin_decay = t_note_off * 0.5

    # 4. Coordenadas de los nodos (X: Tiempo, Y: Amplitud)
    puntos_x = [t_note_on, t_fin_attack, t_fin_decay, t_note_off, t_fin_sonido]
    puntos_y = [0.0, amp_max, amp_sustain_inicio, amp_sustain_final, 0.0]

    # 5. Generar vector de tiempo e interpolar linealmente
    t_env = np.linspace(0, duracion_nota, int(fs * duracion_nota))
    envolvente_lineal = np.interp(t_env, puntos_x, puntos_y)

    # 6. Modulación punto a punto
    min_length = min(len(audio_sintetizado), len(envolvente_lineal))
    audio_modulado = audio_sintetizado[:min_length] * envolvente_lineal[:min_length]

    # Normalización para evitar saturación digital
    if np.max(np.abs(audio_modulado)) > 0:
        audio_modulado = audio_modulado / np.max(np.abs(audio_modulado))

    return audio_modulado, envolvente_lineal

In [ ]:
# =====================================================================
# EJECUCIÓN Y GRAFICACIÓN (CORREGIDO)
# =====================================================================

# Ejecutamos el script con tu criterio de desaceleración para el Release
parametros_perfectos = extraer_parametros_adsr_criterio_usuario(envolvente_extraida, fs)

if parametros_perfectos:
    idx_inicio, idx_pico, idx_fin_decay, idx_note_off, idx_fin = parametros_perfectos["Puntos_Indices"]

    print("==========================================================")
    print("       🎯 CONCORDANCIA ADSR LOGRADA                       ")
    print("==========================================================")
    # CORRECCIÓN DE SINTAXIS: Cambiados los puntos (.) por dos puntos (:) antes del 3f
    print(f" 🔹 Attack (A):  {parametros_perfectos['Attack (s)']:.3f} s")
    print(f" 🔹 Decay (D):   {parametros_perfectos['Decay (s)']:.3f} s")
    print(f" 🔹 Sustain (S): {parametros_perfectos['Sustain (nivel)']:.2f}")
    print(f" 🔹 Release (R): {parametros_perfectos['Release (s)']:.3f} s")
    print("==========================================================")

    plt.figure(figsize=(12, 5))
    plt.plot(tiempo_real, envolvente_extraida, color='darkorange', linewidth=3, label='Envolvente')

    plt.axvline(tiempo_real[idx_inicio], color='green', linestyle='--', label=f"Note On ({tiempo_real[idx_inicio]:.2f}s)")
    plt.axvline(tiempo_real[idx_pico], color='red', linestyle='--', label=f"Fin Attack / Max ({tiempo_real[idx_pico]:.2f}s)")
    plt.axvline(tiempo_real[idx_fin_decay], color='purple', linestyle='--', label=f"Fin Decay ({tiempo_real[idx_fin_decay]:.2f}s)")
    plt.axvline(tiempo_real[idx_note_off], color='blue', linestyle='--', label=f"Note Off / Release ({tiempo_real[idx_note_off]:.2f}s)")
    plt.axvline(tiempo_real[idx_fin], color='black', linestyle='--', label=f"Fin Sonido ({tiempo_real[idx_fin]:.2f}s)")

    plt.xlabel('Tiempo (segundos)')
    plt.ylabel('Amplitud')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.05, 1.5)
    plt.show()

In [ ]:
import numpy as np

# Tomamos los valores absolutos (sin normalizar todavía) para calcular tus parámetros reales
amp_maxima_real = envolvente_extraida[idx_pico]
amp_sustain_real = envolvente_extraida[idx_fin_decay]

# 1. Tu Amplitud Base (A_0) es la amplitud en la etapa de sustain
A_0 = amp_sustain_real

# 2. Tu factor de multiplicación del ataque (k)
# Como Pico = k * A_0  =>  k = Pico / A_0
k = amp_maxima_real / A_0 if A_0 > 0 else 0

# 3. Estimación del Fall Rate (alpha)
# Asumiendo una caída exponencial simple entre el pico y el fin del decay:
# Amp(t) = Amp_pico * exp(-alpha * t)
# alpha = -ln(Amp_decay / Amp_pico) / tiempo_decay
t_decay = adsr_completo["tiempo_decay"]
if t_decay > 0 and amp_sustain_real > 0:
    alpha = -np.log(amp_sustain_real / amp_maxima_real) / t_decay
else:
    alpha = 0.0

print("==========================================================")
print("     📐 PARÁMETROS DE TU MODELO MATEMÁTICO EXTRAÍDOS       ")
print("==========================================================")
print(f"  🔹 Amplitud base (A_0):      {A_0:.4f}")
print(f"  🔹 Multiplicador Ataque (k): {k:.2f}  (El pico llega a {k:.1f} veces A_0)")
print(f"  🔹 Fall Rate (alpha):        {alpha:.2f}")
print("==========================================================")
print(f"  ⏱️ Tiempo de Ataque:         {adsr_completo['tiempo_attack']:.3f} s")
print(f"  ⏱️ Tiempo de Release (R):     {adsr_completo['tiempo_release']:.3f} s")
print("==========================================================")

## Aplicación de envolvente sobre sintesis aditiva

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

def aplicar_envolvente_lineal_dinamica(audio, fs, parametros_adsr, duracion_nota):
    """
    Aplica una envolvente ADSR con un fall rate constante en el Sustain.
    Si la amplitud llega a cero en el Sustain, se mantiene en cero.
    """
    N = len(audio)
    t = np.linspace(0, duracion_nota, N)
    envolvente = np.zeros(N)

    # Extraer tiempos y niveles base
    t_att = parametros_adsr['tiempo_attack']
    t_dec = parametros_adsr['tiempo_decay']
    t_rel = parametros_adsr['tiempo_release']

    amp_max = 1.0  # Asumimos pico de Attack en 1.0 (o normalizado)
    amp_sus_init = parametros_adsr.get('nivel_sustain', 0.7)

    # Obtener o calcular el fall rate (pendiente por segundo, valor negativo)
    # Si no existe en tu diccionario, usamos un valor por defecto (ej: -0.15 de amplitud por segundo)
    fall_rate = parametros_adsr.get('fall_rate', -0.15)
    if fall_rate > 0:
        fall_rate = -fall_rate # Asegurar que sea negativo

    # Nodos de tiempo acumulados
    fin_attack = t_att
    fin_decay = t_att + t_dec
    inicio_release = max(fin_decay, duracion_nota - t_rel)

    # Tiempo disponible real para el Sustain
    tiempo_sustain_max = inicio_release - fin_decay

    # Calcular caída teórica en el sustain
    amp_sustain_final_teorica = amp_sus_init + (fall_rate * tiempo_sustain_max)

    # --- CONSTRUCCIÓN DE LA ENVOLVENTE MUESTRA A MUESTRA ---
    # Variables de control para el gráfico real
    t_fin_sustain_real = inicio_release
    se_extinguio = False

    for i, ti in enumerate(t):
        if ti <= fin_attack:
            # 1. ATTACK (Rampa ascendente)
            envolvente[i] = (amp_max / t_att) * ti if t_att > 0 else amp_max

        elif ti <= fin_decay:
            # 2. DECAY (Caída al nivel inicial de Sustain)
            pct = (ti - fin_attack) / t_dec if t_dec > 0 else 1
            envolvente[i] = amp_max - (amp_max - amp_sus_init) * pct

        elif ti <= inicio_release:
            # 3. SUSTAIN (Caída con pendiente constante 'fall_rate')
            dt_sustain = ti - fin_decay
            amp_actual = amp_sus_init + (fall_rate * dt_sustain)

            if amp_actual > 0:
                envolvente[i] = amp_actual
            else:
                # El sustain se hizo cero antes del Note Off
                envolvente[i] = 0.0
                if not se_extinguio:
                    t_fin_sustain_real = ti  # Guardamos el segundo exacto donde murió
                    se_extinguio = True

        else:
            # 4. RELEASE
            if se_extinguio:
                # Si ya se hizo cero en el sustain, se queda en cero
                envolvente[i] = 0.0
            else:
                # Cae desde donde quedó el sustain hasta cero
                pct_rel = (ti - inicio_release) / t_rel if t_rel > 0 else 1
                amp_al_soltar = amp_sus_init + (fall_rate * tiempo_sustain_max)
                envolvente[i] = max(0.0, amp_al_soltar * (1 - pct_rel))

    # Aplicar la envolvente al audio
    audio_modulado = audio * envolvente

    # Devolvemos también los datos de control para el gráfico
    info_grafico = {
        'fin_attack': fin_attack,
        'fin_decay': fin_decay,
        'inicio_release': inicio_release,
        'se_extinguio': se_extinguio,
        't_extincion': t_fin_sustain_real
    }

    return audio_modulado, envolvente, info_grafico

In [ ]:
from IPython.display import Audio
import matplotlib.pyplot as plt
import numpy as np

# 1. Extraer parámetros del análisis robusto
adsr_analizado = extraer_parametros_adsr_completos(envolvente_extraida, fs)

if adsr_analizado:
    # 2. Aplicar la envolvente corregida con caída constante y control de extinción
    audio_final, env_aplicada, info = aplicar_envolvente_lineal_dinamica(
        audio_aditivo,
        fs=fs,
        parametros_adsr=adsr_analizado,
        duracion_nota=duracion_nota
    )

    # 3. Graficación de control inteligente
    plt.figure(figsize=(12, 5))
    tiempo_final = np.linspace(0, duracion_nota, len(audio_final))

    # Graficar audio y envolvente
    plt.plot(tiempo_final, audio_final, color='silver', alpha=0.5, label='Audio Aditivo Modulado')
    plt.plot(tiempo_final, env_aplicada, color='darkorange', linewidth=3, label='Envolvente Lineal')

    # Líneas guía basadas en los nodos reales devueltos por TU función
    plt.axvline(info['fin_attack'], color='red', linestyle='--', label='Fin Attack')
    plt.axvline(info['fin_decay'], color='purple', linestyle='--', label='Fin Decay')

    # Si se extinguió en el Sustain, marcamos dónde murió; si no, mostramos el Note Off (Release)
    if info['se_extinguio']:
        plt.axvline(info['t_extincion'], color='crimson', linestyle=':', linewidth=2, label='Extinción en Sustain')
    else:
        plt.axvline(info['inicio_release'], color='blue', linestyle='--', label='Inicio Release (Note Off)')

    # Configuración estética del gráfico
    plt.xlabel('Tiempo (segundos)')
    plt.ylabel('Amplitud Normalizada')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.05, duracion_nota + 0.1)
    plt.show()

    # 4. Reproductor directo en el Notebook usando tus datos reales
    display(Audio(audio_final, rate=fs))

else:
    print("Error: No se pudieron extraer los parámetros ADSR de la envolvente analizada.")

Lo que viene aca es xq queria ver la FFT de la resultante, no es relevante para el trabajo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def graficar_comparacion_fft(audio_puro, audio_modulado, fs):
    """
    Calcula y grafica la FFT del audio original vs el audio modulado
    para observar el ensanchamiento espectral y los efectos de los transitorios.
    """
    N = len(audio_modulado)
    # Evitamos problemas de dimensión asegurando el mismo largo
    audio_puro_recortado = audio_puro[:N]

    # 1. Calcular las FFTs
    fft_puro = np.fft.fft(audio_puro_recortado)
    fft_modulado = np.fft.fft(audio_modulado)

    # Vector de frecuencias (solo la mitad positiva)
    frecuencias = np.fft.fftfreq(N, 1/fs)[:N//2]

    # Magnitudes normalizadas en Decibeles (dB) para apreciar mejor los detalles sutiles
    mag_pura = 20 * np.log10(np.abs(fft_puro[:N//2]) + 1e-8)
    mag_modulada = 20 * np.log10(np.abs(fft_modulado[:N//2]) + 1e-8)

    # Normalizamos a 0 dB para que queden en la misma escala visual
    mag_pura -= np.max(mag_pura)
    mag_modulada -= np.max(mag_modulada)

    # 2. Gráfico del Espectro Completo
    plt.figure(figsize=(12, 5))
    plt.plot(frecuencias, mag_pura, color='silver', linewidth=2, label='Audio Aditivo Puro (Estático)')
    plt.plot(frecuencias, mag_modulada, color='darkorange', linewidth=1.5, alpha=0.8, label='Audio Modulado (ADSR Lineal)')

    plt.title('Efecto de la Modulación ADSR en el Espectro de Frecuencias', fontsize=13, fontweight='bold')
    plt.xlabel('Frecuencia (Hz)')
    plt.ylabel('Magnitud Normalizada (dB)')

    # Limitamos el eje X al rango audible de tu instrumento (por ejemplo, hasta 4000 Hz)
    plt.xlim(0, 4000)
    plt.ylim(-60, 5) # Enfocado en los primeros 60 dB de dinámica
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    plt.show()

    # 3. Gráfico de zoom sobre el Primer Armónico (Fundamental)
    # Buscamos el entorno del pico para hacer un zoom automático
    idx_fundamental = np.argmax(mag_modulada)
    f_fund = frecuencias[idx_fundamental]

    plt.figure(figsize=(12, 4))
    plt.plot(frecuencias, mag_pura, color='silver', linewidth=2.5, marker='o', label='Puro')
    plt.plot(frecuencias, mag_modulada, color='darkorange', linewidth=2, marker='.', label='Modulado')

    plt.title(f'Zoom en el Armónico Fundamental (~{f_fund:.1f} Hz) - Ensanchamiento por Ventaneo', fontsize=12)
    plt.xlabel('Frecuencia (Hz)')
    plt.ylabel('Magnitud (dB)')
    plt.xlim(max(0, f_fund - 50), f_fund + 50) # Ventana estrecha de 100 Hz
    plt.ylim(-40, 2)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# Ejecución del análisis espectral
graficar_comparacion_fft(audio_aditivo, audio_final, fs)

###Envolvente exponencial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

def aplicar_envolvente_exponencial_dinamica(audio, fs, parametros_adsr, duracion_nota):
    """
    Aplica una envolvente ADSR con curvas exponenciales y caída dinámica en el Sustain.
    Evita asíntotas infinitas truncando cerca de cero (umbral de silencio).
    """
    N = len(audio)
    t = np.linspace(0, duracion_nota, N)
    envolvente = np.zeros(N)

    # Extraer tiempos y niveles base
    t_att = max(1e-5, parametros_adsr['tiempo_attack']) # Evitar división por cero
    t_dec = max(1e-5, parametros_adsr['tiempo_decay'])
    t_rel = max(1e-5, parametros_adsr['tiempo_release'])

    amp_max = 1.0
    amp_sus_init = parametros_adsr.get('nivel_sustain', 0.7)
    if amp_sus_init <= 0: amp_sus_init = 1e-3 # Evitar ceros en cálculos logarítmicos

    # Nodos de tiempo
    fin_attack = t_att
    fin_decay = t_att + t_dec
    inicio_release = max(fin_decay, duracion_nota - t_rel)
    tiempo_sustain_max = inicio_release - fin_decay

    # Definir la tasa de caída exponencial en el Sustain
    fall_rate_lin = parametros_adsr.get('fall_rate', -0.15)
    fall_factor = np.exp(fall_rate_lin)

    # Constantes de tiempo (Tau) para las curvas
    tau_att = t_att / 5.0
    tau_dec = t_dec / 5.0
    tau_rel = t_rel / 5.0

    UMBRAL_SILENCIO = 0.005

    # Variables de control para el gráfico
    t_fin_sustain_real = inicio_release
    se_extinguio = False
    amp_al_soltar = amp_sus_init * (fall_factor ** tiempo_sustain_max)

    for i, ti in enumerate(t):
        if ti <= fin_attack:
            # 1. ATTACK Exponencial
            envolvente[i] = amp_max * (1 - np.exp(-ti / tau_att))

        elif ti <= fin_decay:
            # 2. DECAY Exponencial
            dt_decay = ti - fin_attack
            envolvente[i] = amp_sus_init + (amp_max - amp_sus_init) * np.exp(-dt_decay / tau_dec)

        elif ti <= inicio_release:
            # 3. SUSTAIN Exponencial Dinámico
            dt_sustain = ti - fin_decay
            amp_actual = amp_sus_init * (fall_factor ** dt_sustain)

            if amp_actual > UMBRAL_SILENCIO:
                envolvente[i] = amp_actual
            else:
                envolvente[i] = 0.0
                if not se_extinguio:
                    t_fin_sustain_real = ti
                    se_extinguio = True
                    amp_al_soltar = 0.0

        else:
            # 4. RELEASE Exponencial
            if se_extinguio:
                envolvente[i] = 0.0
            else:
                dt_release = ti - inicio_release
                envolvente[i] = amp_al_soltar * np.exp(-dt_release / tau_rel)
                if envolvente[i] < UMBRAL_SILENCIO:
                    envolvente[i] = 0.0

    # Aplicar la envolvente al audio
    audio_modulado = audio * envolvente

    info_grafico = {
        'fin_attack': fin_attack,
        'fin_decay': fin_decay,
        'inicio_release': inicio_release,
        'se_extinguio': se_extinguio,
        't_extincion': t_fin_sustain_real
    }

    return audio_modulado, envolvente, info_grafico

In [ ]:
# =====================================================================
# EJECUCIÓN PRINCIPAL: ENVOLVENTE EXPONENCIAL
# =====================================================================

# 1. Extraer parámetros del análisis robusto
adsr_analizado = extraer_parametros_adsr_completos(envolvente_extraida, fs)

if adsr_analizado:
    # 2. Aplicar la NUEVA envolvente exponencial dinámica (Asegurate de haber corrido la función corregida antes)
    audio_final, env_aplicada, info = aplicar_envolvente_exponencial_dinamica(
        audio_aditivo,
        fs=fs,
        parametros_adsr=adsr_analizado,
        duracion_nota=duracion_nota
    )

    # 3. Graficación de control inteligente
    plt.figure(figsize=(12, 5))
    tiempo_final = np.linspace(0, duracion_nota, len(audio_final))

    # Graficar audio y envolvente exponencial
    plt.plot(tiempo_final, audio_final, color='silver', alpha=0.5, label='Audio Aditivo Modulado')
    plt.plot(tiempo_final, env_aplicada, color='crimson', linewidth=3, label='Envolvente Exponencial')

    # Líneas guía basadas en los nodos reales de tiempo
    plt.axvline(info['fin_attack'], color='red', linestyle='--', alpha=0.7, label='Fin Attack')
    plt.axvline(info['fin_decay'], color='purple', linestyle='--', alpha=0.7, label='Fin Decay')

    if info['se_extinguio']:
        plt.axvline(info['t_extincion'], color='black', linestyle=':', linewidth=2, label='Extinción en Sustain')
    else:
        plt.axvline(info['inicio_release'], color='blue', linestyle='--', alpha=0.7, label='Inicio Release (Note Off)')

    # Configuración de pantalla
    plt.xlabel('Tiempo (segundos)')
    plt.ylabel('Amplitud Normalizada')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.25)
    plt.xlim(-0.05, duracion_nota + 0.1)
    plt.ylim(-1.05, 1.05)
    plt.show()

    # 4. Reproductor directo en el Notebook
    display(Audio(audio_final, rate=fs))

else:
    print("Error: No se pudieron extraer los parámetros ADSR de la envolvente analizada.")

### Envolvente por parcial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
from IPython.display import Audio, display

# 1. VERIFICACIÓN: Traemos el audio de la síntesis MIDI
variable_audio_midi = 'audio_sintetizado'

if variable_audio_midi in globals() or variable_audio_midi in locals():
    audio = globals().get(variable_audio_midi, locals().get(variable_audio_midi))
    print(f"✅ ¡Conectado! Audio tomado de la síntesis MIDI para el Espectrograma.")
    print(f"Muestras totales: {len(audio)} | Frecuencia de muestreo (fs): {fs} Hz")

    if audio.ndim > 1:
        audio = audio[:, 0]
else:
    print("❌ ERROR: No encontré 'audio_sintetizado'.")
    print("Por favor, fijate cómo se llama la variable de salida en tu celda de síntesis MIDI.")
    audio = None

# 2. PROCESAMIENTO Y ESPECTROGRAMA (STFT)
if audio is not None:
    # Quitar el valor medio (DC offset)
    audio_limpio = audio - np.mean(audio)

    # --- PARAMETRIZACIÓN CRÍTICA PARA EL PUNTO D ---
    # Usamos una ventana larga (N_fft = 2048 o 4096) para tener excelente resolución en frecuencia
    # y poder separar bien los parciales (las líneas horizontales).
    nperseg = 4096  # Tamaño de la ventana de Hann (equivalente al N_fft anterior)
    noverlap = int(nperseg * 0.75)  # 75% de solapamiento para no perder resolución temporal
    nfft = 4096     # Puntos de la FFT

    # Calcular la STFT
    # f: vector de frecuencias, t: vector de tiempos, Zxx: matriz compleja del espectro
    f, t, Zxx = stft(audio_limpio, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap, nfft=nfft)

    # Obtenemos la magnitud del espectro
    magnitud_stft = np.abs(Zxx)

    # Convertimos la magnitud a Decibeles (dB) para poder ver tanto los parciales fuertes
    # como los más débiles (¡clave para modelar las envolventes después!)
    magnitud_db = 20 * np.log10(magnitud_stft + 1e-6) # Evitamos log(0) con un piso imperceptible

    # 3. GRAFICAR EL ESPECTROGRAMA
    plt.figure(figsize=(12, 6))

    # Usamos pcolormesh para renderizar la matriz de tiempo vs frecuencia
    # shading='gutter' o 'auto' para que alinee bien los ejes
    espectro = plt.pcolormesh(t, f, magnitud_db, cmap='magma', vmin=magnitud_db.max()-60, shading='auto')

    # Personalización de los ejes
    plt.title("Espectrograma de la Evolución Temporal de los Parciales", fontsize=14, pad=15)
    plt.xlabel("Tiempo [segundos]", fontsize=12)
    plt.ylabel("Frecuencia [Hz]", fontsize=12)

    # Acotamos el eje Y para concentrarnos en la zona donde están los parciales audibles (ej: 0 a 4000 Hz)
    plt.ylim(0, 4000)

    # Barra de color para entender la intensidad en dB
    cbar = plt.colorbar(espectro)
    cbar.set_label("Intensidad [dB]", fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.5, color='white') # Grilla sutil blanca para contrastar con 'magma'

    # Guardar en PDF para el informe final
    plt.savefig("espectrograma_parciales.pdf", format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # 4. AUDITORÍA RAPIDA
    print(f"📊 Dimensiones de la matriz del espectrograma: {magnitud_stft.shape}")
    print(f"-> Cantidad de bins de frecuencia: {magnitud_stft.shape[0]}")
    print(f"-> Cantidad de tramas de tiempo (ventanas calculadas): {magnitud_stft.shape[1]}")
    print("\n🔊 Audio completo bajo análisis:")
    display(Audio(audio_limpio, rate=fs))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# =====================================================================
# 1. DEFINICIÓN DE LOS PUNTOS TEMPORALES CLAVE (PARAMETRIZACIÓN)
# =====================================================================
# Basado en tu espectrograma, definimos los momentos donde cambia el sonido
tiempos_clave = np.array([0.0, 0.08, 0.45, 0.95, 1.05])
n_parametros = len(tiempos_clave)

# Diccionario para guardar los parámetros de cada parcial: { f_k: [amp_t0, amp_t1, ...] }
parametros_envolventes = {}

print("==========================================================")
# 'f' y 'magnitud_stft' vienen de la celda de tu espectrograma anterior
# 'fk_finales' es tu lista de frecuencias de los armónicos

for i, fk in enumerate(fk_finales):
    # Encontrás el índice en la STFT que más se acerca a la frecuencia de tu armónico
    idx_frecuencia = np.argmin(np.abs(f - fk))

    # Extraemos la evolución de la amplitud real (en escala lineal, no dB para sintetizar)
    envolvente_cruda = magnitud_stft[idx_frecuencia, :]

    # Creamos una función interpoladora temporal para poder consultar cualquier tiempo
    # t viene del cálculo de la STFT anterior
    interpolador_crudo = interp1d(t, envolvente_cruda, bounds_error=False, fill_value=0.0)

    # Evaluamos la amplitud real del parcial únicamente en nuestros tiempos clave
    amplitudes_clave = interpolador_crudo(tiempos_clave)

    # Forzamos que empiece y termine en 0 para evitar clicks analógicos
    amplitudes_clave[0] = 0.0
    amplitudes_clave[-1] = 0.0

    # Guardamos los 5 parámetros de este parcial
    parametros_envolventes[fk] = amplitudes_clave

print(f"✅ Parametrización completada.")
print(f"Cada parcial ahora se define con solo {n_parametros} puntos en lugar de {len(t)} tramas.")
print("==========================================================")

# =====================================================================
# 2. NUEVA FUNCIÓN DE SÍNTESIS ADITIVA CON ENVOLVENTES POR PARCIAL
# =====================================================================
def sintesis_aditiva_con_envolventes(fk_list, parametros_env, tiempos_c, duracion, fs):
    N_muestras = int(duracion * fs)
    t_vector = np.linspace(0, duracion, N_muestras, endpoint=False)
    audio_sintetizado = np.zeros(N_muestras)

    plt.figure(figsize=(10, 4))

    # Reconstruimos y sumamos cada parcial uno por uno
    for fk in fk_list:
        amps_clave = parametros_env[fk]

        # Reconstruimos la envolvente continua Ak(t) interpolando linealmente los 5 puntos
        interp_envolvente = interp1d(tiempos_c, amps_clave, kind='linear', bounds_error=False, fill_value=0.0)
        Ak_t = interp_envolvente(t_vector)

        # Graficamos algunas envolventes para el informe (las primeras 4 para no saturar)
        if fk in fk_list[:4]:
            plt.plot(t_vector, Ak_t, label=f"Parcial {fk:.1f} Hz")

        # Ecuación del modelo: Ak(t) * sin(2 * pi * fk * t)
        parcial_sintetizado = Ak_t * np.sin(2 * np.pi * fk * t_vector)

        # Acumulamos en el audio final
        audio_sintetizado += parcial_sintetizado

    plt.title("Modelado de Envolventes Parametrizadas $A_k(t)$")
    plt.xlabel("Tiempo [segundos]")
    plt.ylabel("Amplitud")
    plt.legend()
    plt.grid(True, linestyle=':')
    plt.show()

    # Normalización final del audio para evitar saturación (clipping)
    if np.max(np.abs(audio_sintetizado)) > 0:
        audio_sintetizado = audio_sintetizado / np.max(np.abs(audio_sintetizado))

    return audio_sintetizado, t_vector

# =====================================================================
# 3. EJECUCIÓN DEL NUEVO MODELO VARIANTE EN EL TIEMPO
# =====================================================================
audio_aditivo_dinamico, vector_tiempo = sintesis_aditiva_con_envolventes(
    fk_finales,
    parametros_envolventes,
    tiempos_clave,
    duracion=duracion_nota,
    fs=fs
)

print("\n🔊 Escuchá el resultado con Envolventes por Parcial (Punto iii):")
display(Audio(audio_aditivo_dinamico, rate=fs))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

max_parciales_a_mostrar = 4
parciales_seleccionados = fk_finales[:max_parciales_a_mostrar]

plt.figure(figsize=(12, 6))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, fk in enumerate(parciales_seleccionados):
    # 1. Extraer la envolvente original (cruda)
    idx_frecuencia = np.argmin(np.abs(f - fk))
    envolvente_cruda = magnitud_stft[idx_frecuencia, :]

    # 2. Reconstruir la envolvente parametrizada
    amps_clave = parametros_envolventes[fk]
    interp_envolvente = interp1d(tiempos_clave, amps_clave, kind='linear', bounds_error=False, fill_value=0.0)

    t_continuo = np.linspace(0, duracion_nota + 0.1, 1000) # Un cachito más de la duración
    envolvente_parametrizada = interp_envolvente(t_continuo)

    # --- TRUCO DE ESCALADO: NORMALIZACIÓN INDIVIDUAL ---
    # Buscamos el máximo de la original para normalizar ambas curvas por igual
    max_original = np.max(envolvente_cruda)

    if max_original > 0:
        envolvente_cruda_norm = envolvente_cruda / max_original
        envolvente_parametrizada_norm = envolvente_parametrizada / max_original
    else:
        envolvente_cruda_norm = envolvente_cruda
        envolvente_parametrizada_norm = envolvente_parametrizada
    # ---------------------------------------------------

    color = colores[idx % len(colores)]

    # Graficar curvas normalizadas
    plt.plot(t, envolvente_cruda_norm, linestyle='--', alpha=0.4, color=color,
             label=f"Original (Norm) - {fk:.1f} Hz")
    plt.plot(t_continuo, volver_envolvente_norm := envolvente_parametrizada_norm, linestyle='-', linewidth=2.5, color=color,
             label=f"Parametrizada (Norm) - {fk:.1f} Hz")

# Configuración estética y ZOOM
plt.title("Tendencias Escalas y Normalizadas de las Envolventes por Parcial", fontsize=14, pad=15)
plt.xlabel("Tiempo [segundos]", fontsize=12)
plt.ylabel("Amplitud Normalizada (0 a 1)", fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)

# --- QUITAR SILENCIO DE LOS EJES ---
plt.xlim(-0.02, 1.1)  # Zoom exacto en la zona de actividad de la nota (0 a 1.1s)
plt.ylim(-0.05, 1.05) # Ajuste vertical limpio para el rango 0-1

# Posicionar la leyenda afuera
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

# Dibujar las líneas de los tiempos clave dentro del zoom
for tc in tiempos_clave:
    if tc <= 1.1:
        plt.axvline(x=tc, color='gray', linestyle=':', alpha=0.7, linewidth=1)
        plt.text(tc, -0.04, f"{tc}s", color='gray', fontsize=9, ha='center')

plt.tight_layout()
plt.savefig("comparativa_envolventes_zoom.pdf", format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Diccionarios dinámicos: ahora guardamos tanto los tiempos como las amplitudes por separado para cada fk
tiempos_personalizados = {}
amplitudes_personalizadas = {}

print("==========================================================")
print(" Extraction de Envolventes Independientes por Parcial ")
print("==========================================================")

for fk in fk_finales:
    idx_frecuencia = np.argmin(np.abs(f - fk))
    envolvente_cruda = magnitud_stft[idx_frecuencia, :]

    # --- DETECCIÓN DE PUNTOS CRÍTICOS INDEPENDIENTES ---
    # 1. Inicio
    t_start = 0.0
    a_start = 0.0

    # 2. Pico de Ataque (Buscamos el máximo real de ESTE parcial)
    idx_pico = np.argmax(envolvente_cruda)
    t_attack = t[idx_pico]
    a_attack = envolvente_cruda[idx_pico]

    # 3. Punto medio / Transitorio (Buscamos la zona del re-disparo cerca de 0.45s)
    idx_medio = np.argmin(np.abs(t - 0.45))
    t_decay = 0.45
    a_decay = envolvente_cruda[idx_medio]

    # 4. Fin del cuerpo (Justo antes del corte en 0.95s)
    idx_cuerpo = np.argmin(np.abs(t - 0.95))
    t_sustain = 0.95
    a_sustain = envolvente_cruda[idx_cuerpo]

    # 5. Apagado total
    t_release = 1.05
    a_release = 0.0

    # Agrupamos los vectores de tiempo y amplitud Propios de este parcial
    tiempos_personalizados[fk] = np.array([t_start, t_attack, t_decay, t_sustain, t_release])
    amplitudes_personalizadas[fk] = np.array([a_start, a_attack, a_decay, a_sustain, a_release])

print("✅ Envolventes personalizadas calculadas con éxito.")

In [ ]:
max_parciales_a_mostrar = 4
parciales_seleccionados = fk_finales[:max_parciales_a_mostrar]

plt.figure(figsize=(12, 6))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, fk in enumerate(parciales_seleccionados):
    idx_frecuencia = np.argmin(np.abs(f - fk))
    envolvente_cruda = magnitud_stft[idx_frecuencia, :]

    # Recuperamos sus parámetros únicos
    t_propios = tiempos_personalizados[fk]
    a_propios = amplitudes_personalizadas[fk]

    # Interpolamos usando sus propios tiempos
    interp_envolvente = interp1d(t_propios, a_propios, kind='linear', bounds_error=False, fill_value=0.0)
    t_continuo = np.linspace(0, 1.1, 1000)
    envolvente_parametrizada = interp_envolvente(t_continuo)

    # Normalización para el gráfico
    max_original = np.max(envolvente_cruda)
    if max_original > 0:
        envolvente_cruda_norm = envolvente_cruda / max_original
        envolvente_parametrizada_norm = envolvente_parametrizada / max_original

    color = colores[idx % len(colores)]

    # Graficar
    plt.plot(t, envolvente_cruda_norm, linestyle='--', alpha=0.3, color=color,
             label=f"Original - {fk:.1f} Hz")
    plt.plot(t_continuo, envolvente_parametrizada_norm, linestyle='-', linewidth=2.5, color=color,
             label=f"Modelo Independiente - {fk:.1f} Hz")

    # Dibujamos circulitos en los nodos clave para ver dónde se clavaron los parámetros
    # (Dividimos por max_original para que coincida con la escala del gráfico)
    plt.scatter(t_propios, a_propios / max_original, color=color, s=40, zorder=5)

plt.title("Modelo Final: Envolventes con Parámetros Temporales Independientes", fontsize=14, pad=15)
plt.xlabel("Tiempo [segundos]", fontsize=12)
plt.ylabel("Amplitud Normalizada (0 a 1)", fontsize=12)
plt.xlim(-0.02, 1.1)
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
def sintesis_aditiva_final(fk_list, t_pers, a_pers, duracion, fs):
    N_muestras = int(duracion * fs)
    t_vector = np.linspace(0, duracion, N_muestras, endpoint=False)
    audio_sintetizado = np.zeros(N_muestras)

    for fk in fk_list:
        t_propios = t_pers[fk]
        a_propios = a_pers[fk]

        # Cada parcial se reconstruye con su propia línea temporal de interpolación
        interp_envolvente = interp1d(t_propios, a_propios, kind='linear', bounds_error=False, fill_value=0.0)
        Ak_t = interp_envolvente(t_vector)

        # Sumamos al modelo aditivo
        audio_sintetizado += Ak_t * np.sin(2 * np.pi * fk * t_vector)

    if np.max(np.abs(audio_sintetizado)) > 0:
        audio_sintetizado = audio_sintetizado / np.max(np.abs(audio_sintetizado))

    return audio_sintetizado

# Ejecución final
audio_aditivo_perfecto = sintesis_aditiva_final(fk_finales, tiempos_personalizados, amplitudes_personalizadas, duracion_nota, fs)
display(Audio(audio_aditivo_perfecto, rate=fs))